<a href="https://colab.research.google.com/github/manimalakumar/RAG-with-without-advance-techniques/blob/main/RAG_Retriever_Reranker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compare RAG with Reranking vs without Reranking


In this notebook, let's build a pipeline for a response generating LLM, RAG, embedding and with or without reranking. Then compare the answers of basic RAG pipeline with the enhanced RAG pipeline with reranking for the same input prompts.

By prioritizing the right documents, reranking increases the likelihood of providing the LLM with the best context, which improves the quality of generated responses.

While both setups will use a context length of 5 documents and the same response generating LLM, the RAG with reranking, provides more relevant documents to the response generating LLM in the applicaiton, thus improving answer accuracy and relevancy.

To rerun the notebook, audience will need API keys for HuggingFace (dataset), Nvidia developers (retriever and reranker models) and OpenAI (generator LLM).

## Installation

Install packages

In [ ]:
!pip install nvidia-haystack datasets

## Dataset

Load [ms_marco v1.1 dataset](https://huggingface.co/datasets/microsoft/ms_marco) from Hugging Face. It is a question/answer dataset.

In [ ]:
import datasets
from datasets import load_dataset

datasets.config.HF_DATASETS_TRUST_REMOTE_CODE = True
data = load_dataset('ms_marco', 'v1.1')


Check an entry to understand the data structure. In ms_marco, each entry includes a  query, passage_text (that is context in our application), url and ground-truth answers.

In [ ]:
data["validation"][1]

{'answers': ['$21,550 per year',
  'The average hourly wage for a bartender is $10.36 and the average yearly take-home is $21,550.'],
 'passages': {'is_selected': [0, 1, 0, 0, 0, 0, 0, 0],
  'passage_text': ['A bartender’s income is comprised mostly of tips– 55% to be exact. In some states, employers aren’t even required to pay their bartenders the minimum wage and can pay as low as $2.13 per hour, and they depend on their tips almost entirely. Bartending can be a lot of things. For some it is exciting, for others exhausting. At times there is a lot of fun to be had, at others it is rather dull. But for the most part, bartending is almost always rewarding in the financial sense, as long as you stick with it.',
   'According to the Bureau of Labor Statistics, the average hourly wage for a bartender is $10.36, and the average yearly take-home is $21,550. Bartending can be a lot of things. For some it is exciting, for others exhausting. At times there is a lot of fun to be had, at others 

Convert the ms_marco dataset entries into Haystack Documents. Context/passage_text is put into chunks and url is used as meta info in the Haystack Document object. A chunk function is created - basic fixed size chunk with overlap to avoid abrupt information loss at truncation of paragraphs. passage_text is split into chunks. url is stored in metadata.

In [ ]:
from haystack.dataclasses.document import Document

def convert_dataset(data):

    doc_chunks = []
    for item in data:
        # Collect the relevant content
        context_dict = {item['passages']["url"][i]: item['passages']["passage_text"][i] for i in range(len(item['passages']["url"]))}

        # Convert to Haystack Documents
        for k, v in context_dict.items():
            content = ''.join(v).strip()
            doc_chunks.append(Document(content=content, meta={"title":k}))

    return doc_chunks # Added return statement

In [ ]:
documents = convert_dataset(data["validation"])
print(documents[0])

Document(id=84bfab3766f315b522b6a4ee6c646ef15bff297dcd870ce2428967c89e6517fb, content: 'The average Walgreens salary ranges from approximately $15,000 per year for Customer Service Associa...', meta: {'title': 'http://www.indeed.com/cmp/Walgreens/salaries'})


## Embedding and Indexing Documents

Embed and Index the documents. Instead of RAG Vector DB, this notebook uses RAG on the fly approach for simplicity and for demo purpose.  Need an NVIDIA API catalog api key to use an AI model for embedding. I signed up at https://build.nvidia.com/nvidia to generate the NVIDIA API catalog key to use NVIDIA AI Models for embedding now and in future cells for other AI Model actions. I have got free credits and terms may change. Set this key value as `"NVIDIA_API_KEY"` environment variable. OPENAI_API_KEY is needed for response generating LLM model. HF_TOKEN is needed for Hugging Face Dataset.

In [ ]:
import os
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = ""
os.environ["OPENAI_API_KEY"] = ""
os.environ["HF_TOKEN"] = ""

Next, create a pipeline and embed and index the documents. For embeddings, we'll use the **llama-3_2-nv-embedqa-1b-v2** model

In [ ]:
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses.document import Document
from haystack.components.writers import DocumentWriter
from haystack_integrations.components.embedders.nvidia import NvidiaDocumentEmbedder
from haystack import Pipeline
from haystack.document_stores.types import DuplicatePolicy
from haystack.components.preprocessors import DocumentSplitter

document_store = InMemoryDocumentStore()
embedder = NvidiaDocumentEmbedder(model="nvidia/llama-3.2-nv-embedqa-1b-v2",
                                  api_url="https://integrate.api.nvidia.com/v1")

indexing_pipeline = Pipeline()
indexing_pipeline.add_component(instance=DocumentSplitter(split_length=350, split_overlap=50), name="splitter")
indexing_pipeline.add_component(instance=embedder, name="embedder")
indexing_pipeline.add_component(instance=DocumentWriter(document_store=document_store, policy=DuplicatePolicy.SKIP), name="writer")
indexing_pipeline.connect("splitter", "embedder")
indexing_pipeline.connect("embedder.documents", "writer.documents")

#indexing_pipeline.run({"splitter":{"documents": documents[:500]}}) # no need to index all documents
indexing_pipeline.run({"splitter":{"documents": documents}})


print(document_store.count_documents())

Calculating embeddings: 100%|██████████| 1949/1949 [12:21<00:00,  2.63it/s]


61400


## RAG with Reranking
Now create a RAG pipeline with a reranking model`nvidia/llama-3.2-nv-rerankqa-1b-v2` model. Set the `top_k` value of retriever to 30 and of reranker to 5. Thus, the code first retrieves 30 docs but only pass the 5 most relevant documents as context to the LLM.

So far input data processing is done meaning context is stored. For response generation, to user query (my query in this case), let's use the `meta/llama3-70b-instruct model`.

In [ ]:
from haystack import Pipeline
from haystack.utils.auth import Secret
from haystack.components.builders import PromptBuilder
from haystack_integrations.components.embedders.nvidia import NvidiaTextEmbedder
from haystack_integrations.components.generators.nvidia import NvidiaGenerator
from haystack_integrations.components.rankers.nvidia import NvidiaRanker
from haystack.components.retrievers import InMemoryEmbeddingRetriever

from haystack.components.generators import OpenAIGenerator

#this model we used for embedding, shown to list the entire pipeline
embedder = NvidiaTextEmbedder(model="nvidia/llama-3.2-nv-embedqa-1b-v2",
                             api_url="https://integrate.api.nvidia.com/v1")

retriever = InMemoryEmbeddingRetriever(document_store=document_store, top_k=30)
reranker = NvidiaRanker(
    model="nvidia/llama-3.2-nv-rerankqa-1b-v2",
    top_k=5
)
prompt = """Answer the question given the context.
Question: {{ query }}
Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}
Answer:
"""
prompt_builder = PromptBuilder(template=prompt)

generator = OpenAIGenerator(
    api_key=Secret.from_token(os.getenv("OPENAI_API_KEY")),
    model= "gpt-4.1-nano",
    generation_kwargs={"temperature": 0.7},
)
enhanced_rag = Pipeline()
enhanced_rag.add_component("embedder", embedder)
enhanced_rag.add_component("retriever", retriever)
enhanced_rag.add_component("reranker", reranker)
enhanced_rag.add_component("prompt_builder", prompt_builder)
enhanced_rag.add_component("generator", generator)

enhanced_rag.connect("embedder.embedding", "retriever.query_embedding")
enhanced_rag.connect("retriever", "reranker")
enhanced_rag.connect("reranker.documents", "prompt_builder.documents")
enhanced_rag.connect("prompt_builder", "generator")

🚅 Components
  - embedder: NvidiaTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - reranker: NvidiaRanker
  - prompt_builder: PromptBuilder
  - generator: OpenAIGenerator
🛤️ Connections
  - embedder.embedding -> retriever.query_embedding (List[float])
  - retriever.documents -> reranker.documents (list[Document])
  - reranker.documents -> prompt_builder.documents (List[Document])
  - prompt_builder.prompt -> generator.prompt (str)

Let's run the pipeline with some prompts (questions) and compare the answers:

---



In [ ]:
question = "What is the cause of COVID-19?" # correct answer we can all relate to
better_results = enhanced_rag.run({
    "embedder": {"text": question},
    "reranker": {"query": question},
    "prompt_builder": {"query": question}
})
print (better_results)

{'embedder': {'meta': {'usage': {'prompt_tokens': 12, 'total_tokens': 12}}}, 'generator': {'replies': ['COVID-19 is caused by a pathogen, specifically a virus called the severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2).'], 'meta': [{'model': 'gpt-4.1-nano-2025-04-14', 'index': 0, 'finish_reason': 'stop', 'usage': {'completion_tokens': 29, 'prompt_tokens': 314, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}}]}}


## Basic RAG Pipeline
For comparison, first let's define a basic pipeline (without a reranker) and see the result for the same questions used in basic RAG. The same questions will be used in RAG + reranker.

In [ ]:
from haystack import Pipeline
from haystack.utils.auth import Secret
from haystack.components.builders import PromptBuilder
from haystack_integrations.components.embedders.nvidia import NvidiaTextEmbedder
#from haystack_integrations.components.generators.nvidia import NvidiaGenerator
from haystack.components.retrievers import InMemoryEmbeddingRetriever

embedder = NvidiaTextEmbedder(model="nvidia/llama-3.2-nv-embedqa-1b-v2",
                              api_url="https://integrate.api.nvidia.com/v1")

retriever = InMemoryEmbeddingRetriever(document_store=document_store, top_k=5)
prompt = """Answer the question given the context.
Question: {{ query }}
Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}
Answer:
"""
prompt_builder = PromptBuilder(template=prompt)
generator = OpenAIGenerator(
    api_key=Secret.from_token(os.getenv("OPENAI_API_KEY")),
    model= "gpt-4.1-nano",
    generation_kwargs={"temperature": 0.7},
)
rag = Pipeline()
rag.add_component("embedder", embedder)
rag.add_component("retriever", retriever)
rag.add_component("prompt_builder", prompt_builder)
rag.add_component("generator", generator)
rag.connect("embedder.embedding", "retriever.query_embedding")
rag.connect("retriever", "prompt_builder.documents")
rag.connect("prompt_builder", "generator")

🚅 Components
  - embedder: NvidiaTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt_builder: PromptBuilder
  - generator: OpenAIGenerator
🛤️ Connections
  - embedder.embedding -> retriever.query_embedding (List[float])
  - retriever.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> generator.prompt (str)

In [ ]:
question = "What is the cause of COVID-19?" # corrct answer we all can relate to
results = rag.run({
    "embedder": {"text": question},
    "prompt_builder": {"query": question}
})
print (results)

{'embedder': {'meta': {'usage': {'prompt_tokens': 12, 'total_tokens': 12}}}, 'generator': {'replies': ['COVID-19 is caused by the novel coronavirus known as SARS-CoV-2.'], 'meta': [{'model': 'gpt-4.1-nano-2025-04-14', 'index': 0, 'finish_reason': 'stop', 'usage': {'completion_tokens': 17, 'prompt_tokens': 480, 'total_tokens': 497, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}}]}}
